- Dados: Focos de calor do INPE na frequência anual - https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/anual/EstadosBr_sat_ref/
- Código realizado por: Enrique V. Mattos - 21/05/2024

# **1° Passo:** Preparando ambiente

In [ ]:
# instalando bibliotecas
!pip install -q ultraplot cartopy salem rasterio

# importa bibliotecas
import numpy as np
import ultraplot as uplt
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import time
from zipfile import ZipFile
import salem
from datetime import datetime
import glob
import xarray as xr
import os
import cartopy.crs as ccrs
import warnings
warnings.filterwarnings('ignore')

# monta drive
from google.colab import drive
drive.mount('/content/drive')

# caminho do drive
dir = '/content/drive/MyDrive/5_EXTENSAO/02_prefeitura_analise_periodo_chuvoso_2024_2025'

# leitura do shapefile do Brasil
shapefile_brasil = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/brasil/BRAZIL.shp')

# url dos shapefiles
url = 'https://github.com/evmpython/shapefile/raw/main/'

# leitura do shapefile com a biblioteca SALEM
shp = salem.read_shapefile(f'{url}itajuba/itajuba.shp')
itajuba = salem.read_shapefile(f'{url}itajuba/itajuba.shp')
mg = salem.read_shapefile(f'{url}estado_MG/MG_UF_2019.shp')

# limites do Brasil
lonmin, lonmax, latmin, latmax = -75.0, -33.0, -35.0, 7.0

# **PARTE 1):** Leitura dos dados de focos de calor

##Baixando os dados. Baixa os dados de outubro do ano anterior até março do ano seguinte. Por exemplo: out/2024 à mar/2025.

In [ ]:
#==============================================================#
#                         2023 e 2025
#==============================================================#
# Exemplo: focos_mensal_br_202501.csv

# ano atual - ano em andamento
anoi, mesi, anof, mesf = '2024','10','2025','03'

# ftp dos dados mensais
url = f'https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/mensal/Brasil/'

# loop dos meses
ano_mesi = f'{anoi}{mesi}01'
ano_mesf = f'{str(anof)}{str(int(mesf)+1).zfill(2)}01'

for data in pd.date_range(ano_mesi, ano_mesf, freq='1M'):

    # estrai ano e mês
    ano = data.strftime('%Y')
    mes = data.strftime('%m')

    # baixa o arquivo
    #if ano == '2023': filename = f'{url}focos_mensal_br_{ano}{mes}.zip'
    if ano == '2024': filename = f'{url}focos_mensal_br_{ano}{mes}.csv'
    if ano == '2025': filename = f'{url}focos_mensal_br_{ano}{mes}.csv'

    !wget {filename}

##Lendo os dados de `outubro/2024` à `março/2025`

In [ ]:
# lista os arquivos
files = sorted(glob.glob('/content/focos_mensal_br*'))

# loop de cada arquivo da lista files
df = pd.DataFrame()
for file in files:

    # nome do arquivo
    basename = os.path.basename(os.path.splitext(file)[0])
    print('Processando ===>>>>', basename)

    # extrai a data da imagem
    ano, mes = basename[16:20], basename[20:22]

    # leitura da tabela
    #if ano=='2023': df0 = pd.read_csv(file, compression='zip')
    #if ano=='2024': df0 = pd.read_csv(file)
    df0 = pd.read_csv(file)

    # junta a tabela que foi lida com a anterior
    df = pd.concat([df, df0], ignore_index=True)

# reposiciona as colunas
df = df[['data_hora_gmt','satelite','lat','lon','municipio','estado','bioma']]

# renomeia coluna
#df.rename(columns={'data_pas': 'data'}, inplace=True)

#transforma a coluna "data_hora_gmt" para o formato "datetime"
df['data_hora_gmt'] = pd.to_datetime(df['data_hora_gmt'])

# seta a coluna "data_hora_gmt" como o índice da tabela
df.set_index('data_hora_gmt', inplace=True)

# mostra o dataframe
df

##Selecionando os dados de interesse

In [ ]:
df.head(2)

In [ ]:
# seleciona para o estado de MG
df = df[ (df['estado']=='MINAS GERAIS') ]
df.head(2)

In [ ]:
# satélites utilizados
df['satelite'].unique()

In [ ]:
# seleciona para o satélite
#name_satelite = 'AQUA_M-T'
#name_satelite = 'GOES-16'
name_satelite = 'TODOS SATELITES'

if name_satelite == 'AQUA_M-T': df = df[ (df['satelite']=='AQUA_M-T') ]
if name_satelite == 'GOES-16': df = df[ (df['satelite']=='GOES-16') ]
if name_satelite == 'TODOS SATELITES': df = df
df

# **PARTE 2):** Processamento

## Função

In [ ]:
# Função que calcula o índice i e j da localização do foco
def index(longitudes_matriz, latitudes_matriz, lon_foco, lat_foco):

    ''' Função para calcular o índice (i e j) do pixel de uma matriz que o relâmpago pertence

    Parâmetros:
               longitudes_matriz (array): array de uma dimensão das longitudes da matriz em graus
               latitudes_matriz (array): array de uma dimensão das latitudes da matriz em graus
               lon_raio (float): valor da longitude do foco em graus
               lat_raio (float): valor da latitude do foco em graus

    Retorna:
            indice_lat_raio (float): índice da latitude (ou seja, da linha) do pixel da matriz que o foco pertence
            indice_lon_raio (float): índice da longitude (ou seja, da coluna) do pixel da matriz que o foco pertence
    '''

    # calcula a diferença entre as lats/lons da matriz e a latitude/longitude do foco
    distancia_lon = (longitudes_matriz - lon_foco)**2
    distancia_lat = (latitudes_matriz - lat_foco)**2

    # índice da longitude e latitude do foco
    indice_lon_foco = np.nonzero(distancia_lon == np.min(distancia_lon))
    indice_lat_foco  = np.nonzero(distancia_lat == np.min(distancia_lat))

    # retorna os valores dos índices calculados
    return indice_lat_foco, indice_lon_foco

##Processamento

In [ ]:
# mostra os dados que serão interpolados para grade
df.head()

In [ ]:
# limites do estado de MG
lonmin, lonmax, latmin, latmax = -52., -39., -23., -14.

# Espaçamento da grade
delta = 20/100.   # grade com 10 km de resolução espacial

# Montando a grade
lons = np.arange(lonmin, lonmax, delta)
lats = np.arange(latmax, latmin, -delta)

# Quantidade de pontos para longitude e latitude
nlon = len(lons)
nlat = len(lats)

# Loop dos meses
for data in pd.date_range('20241001', '20250331', freq='1M'):

    print('Processando ===>>>', data)

    # estrai ano e mês
    ano = data.strftime('%Y')
    mes = data.strftime('%m')

    # seleciona o mês
    df_selec = df.loc[f'{ano}-{mes}']

    # gera matriz de raios
    focos_lon, focos_lat = df_selec['lon'].values, df_selec['lat'].values

    # interpolando para ponto de grade
    focos = np.zeros((nlat, nlon))

    # loop em cada longitude e latitude da lista
    for lonfoco, latfoco in zip(focos_lon, focos_lat):

        # função que extrai a qual pixel aquele relâmpago pertence
        lin, col = index(lons, lats, lonfoco, latfoco)

        # soma os relâmpagos por pixel
        focos[lin,col]+=1

    # gera arquivo netcdf
    data_vars = {'focos':(('lat', 'lon'), focos, {'units': 'ocorrências/10km²', 'long_name':'Focos de Calor'})}
    coords = {'lat': lats, 'lon': lons, 'time': pd.to_datetime(f'{ano}-{mes}')}
    ds = xr.Dataset(data_vars=data_vars, coords=coords)
    ds.to_netcdf(f'{dir}/output/01_FOCOS_CALOR/focos_mensal_MG_{name_satelite.replace(" ", "")}_{ano}_{mes}.nc')

    print('Máximo por pixel', np.max(focos), '\n', 'Máximo do Estado', np.sum(focos), '\n')

# **PARTE 3):** Plota figuras individuais - `MAPA DE DENSIDADE`

In [ ]:
files = sorted(glob.glob(f'{dir}/output/*nc'))
#files = files[0:1]
files

In [ ]:
ds

In [ ]:
%%time
# loop dos arquivos mensais
for file in files:

    # leitura do arquivo netcdf
    ds = xr.open_dataset(file)

    # nome do arquivo
    basename = os.path.basename(os.path.splitext(file)[0])

    # extrai a data da imagem
    ano, mes = basename[6:10], basename[11:14]

    print('Processando ===>>>',  ano, mes)

    # cria a moldura da figura
    fig, ax = uplt.subplots(axwidth=6, tight=True, proj='pcarree')

    # limites de MG
    lonmin, lonmax, latmin, latmax = -52., -39., -23., -14.

    # define formato da figura
    ax.format(coast=False, borders=False, innerborders=False,
              labels=False, latlines=5, lonlines=10,
              latlim=(latmin, latmax), lonlim=(lonmin, lonmax),
              title=f'{ano}-{mes}',
              titleloc='c',
              titleweight='bold',
              titlecolor='black',
              small='20px', large='25px',
              linewidth=0, grid=False)

    # plota subtítulo
    #ax.text(lonmin, latmax-1.0, f'{mes} de {ano}', color='grey', fontsize=10)

    # plota mapa
    map1 = ax.contourf(ds['lon'],
                       ds['lat'],
                       ds['focos'][:,:].salem.roi(shape=mg),
                       extend='max',
                       cmap='lajolla',
                       vmin=0.01, vmax=10,
                       levels=uplt.arange(0.01, 10, 1),
                       colorbar='ll',
                       colorbar_kw={'label': 'Fonte: INPE/Pixel: 20km',
                                    'length': 18,
                                    'frameon': False,
                                    'ticklabelsize': 10,
                                    'labelsize': 7,
                                    'width': 2,
                                    'ticks': 1})

    # plota contorno dos Estados
    mg.plot(edgecolor='black', facecolor='none', linewidth=0.5, alpha=1, ax=ax)
    itajuba.plot(edgecolor='red', facecolor='none', linewidth=0.5, alpha=1, ax=ax)

    print(f'{dir}output/mapa_focos_{ano}_{mes}.png')

# salva figura
fig.save(f'{dir}/output/mapa_focos_{ano}_{mes}.png', dpi=300, bbox_inches='tight')

# **PARTE 4):** Plota painel de figuras - `MAPA DE DENSIDADE`

In [ ]:
ds

In [ ]:
# lista dos arquivos mensais
files = sorted(glob.glob(f'{dir}/output/01_FOCOS_CALOR/focos_mensal_MG_TODOSSATELITES*nc'))
#files = files[0:6]

# meses
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

# cria a moldura da figura
fig, ax = uplt.subplots(figsize=(18, 12.5), nrows=2, ncols=3, tight=True, proj='pcarree', sharex=True, sharey=True)

# formatação dos eixos
ax.format(coast=False, borders=False, innerborders=False,
          labels=False, latlines=5, lonlines=10,
          latlim=(latmin, latmax), lonlim=(lonmin, lonmax),
          suptitle=f'FOCOS DE CALOR - {name_satelite}',
          small='20px', large='35px',
          linewidth=0, grid=False)

# loop dos anos
for i, file in enumerate(files):

    # leitura do arquivo netcdf
    ds = xr.open_dataset(file)

    # nome do arquivo
    basename = os.path.basename(os.path.splitext(file)[0])

    # extrai a data da imagem
    ano, mes = basename[31:35], basename[36:38]

    print('Processando ===>>>',  ano, mes)

    # coloca "NaN" onde os "focos=0"
    condicao = ds['focos'][:,:] == 0
    ds['focos'][:,:] = np.where(condicao, np.nan, ds['focos'][:,:])

    # Ou
    # substituir zeros por NaN na variável 'raio'
    #ds['focos'] = ds['focos'].where(ds['focos'] != 0, np.nan)

    # plota figura
    map1 = ax[i].contourf(ds['lon'],
                          ds['lat'],
                          ds['focos'][:,:].salem.roi(shape=mg),
                          cmap='lajolla',
                          vmin=0.0, vmax=30,
                          levels=uplt.arange(0.0, 30, 5),
                          extend='max')

    # total de focos no estado de MG
    total = np.sum(ds['focos'][:,:].salem.roi(shape=mg))
    total = str(int(total.values))

    # plota titulo de cada figura
    ax[i].format(title=f'{meses[int(mes)-1]}/{ano}={total}', labels = False, titleloc='c', titlecolor='bright blue', fontsize=20)

    # plota contorno de MG e Itajubá
    mg.plot(edgecolor='black', facecolor='none', linewidth=1.2, alpha=1, ax=ax[i])
    itajuba.plot(edgecolor='red', facecolor='none', linewidth=0.5, alpha=1, ax=ax[i])

# informação na figura
ax[3].annotate('Prof. Enrique Mattos/UNIFEI\ngithub.com/evmpython', xy=(lonmin, latmin-4.0), fontsize=15, color='black')

# plota barra de cores da figura
fig.colorbar(map1, loc='b', label='focos/mês*400km$^2$\nFonte: INPE/Pixel: 20km', ticks=5, ticklabelsize=22, labelsize=22, space=0.5, length=0.60, width=0.4)

# salva figura
fig.savefig(f'{dir}/output/Fig_2a_focos_painel_MG_{name_satelite}.jpg', transparent=True, dpi=300, bbox_inches="tight", pad_inches=0.1)

# **PARTE 5):** Plota painel de figuras - `MAPA DE PONTOS`

In [ ]:
df

In [ ]:
# meses
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

# cria a moldura da figura
fig, ax = uplt.subplots(figsize=(18, 12.5), nrows=2, ncols=3, tight=True, proj='pcarree', sharex=True, sharey=True)

# formatação dos eixos
ax.format(coast=False, borders=False, innerborders=False,
          labels=False, latlines=5, lonlines=10,
          latlim=(latmin, latmax), lonlim=(lonmin, lonmax),
          suptitle=f'Focos de Calor - {name_satelite}',
          small='20px', large='35px',
          linewidth=0, grid=False)

# Loop dos meses
for i, data in enumerate(pd.date_range('20241001', '20250331', freq='1M')):

    print('Processando ===>>>', data)

    # estrai ano e mês
    ano = data.strftime('%Y')
    mes = data.strftime('%m')

    # seleciona o mês
    df_selec = df.loc[f'{ano}-{mes}']

    # plota figura
    ax[i].scatter(df_selec['lon'].values, df_selec['lat'].values, transform=ccrs.PlateCarree(), marker='x', s=20, color='bright red')

    # total de focos no estado de MG
    total = df_selec.shape[0]

    # plota titulo de cada figura
    ax[i].format(title=f'{meses[int(mes)-1]}/{ano}={total}', labels = False, titleloc='c', titlecolor='bright blue', fontsize=20)

    # plota contorno de MG e Itajubá
    mg.plot(edgecolor='black', facecolor='none', linewidth=1.2, alpha=1, ax=ax[i])
    itajuba.plot(edgecolor='bright blue', facecolor='none', linewidth=0.5, alpha=1, ax=ax[i])

# informação na figura
ax[3].annotate('Prof. Enrique Mattos/UNIFEI\ngithub.com/evmpython', xy=(lonmin, latmin+0.2), fontsize=12, color='black')

# plota barra de cores da figura
#fig.colorbar(map1, loc='b', label='focos/mês*400km$^2$\nFonte: INPE/Pixel: 20km', ticks=1, ticklabelsize=22, labelsize=22, space=0.5, length=0.60, width=0.4)

# salva figura
fig.savefig(f'{dir}/output/Fig_2b_focos_painel_{name_satelite}_MG.jpg', transparent=True, dpi=300, bbox_inches="tight", pad_inches=0.1)